# HMI Image Stacking Example

Demonstrates solar rotation-corrected image stacking for HMI magnetograms using the egghouse stacking module.

This example covers:
1. Snodgrass differential rotation model
2. StreamingStackAccumulator for memory-efficient processing
3. stack_with_rotation_correction for rotation-corrected stacking
4. Real HMI data download using sunpy Fido (optional)

Requires: `pip install "egghouse[sdo]"`

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
from scipy.ndimage import shift as scipy_shift

# Import stacking module
from egghouse.sdo import (
    snodgrass_rotation_rate,
    solar_rotation_shift,
    StreamingStackAccumulator,
    stack_with_rotation_correction,
    SNODGRASS_A,
    SNODGRASS_B,
    SNODGRASS_C,
    SOLAR_ROTATION_PERIOD,
    HMI_CADENCE_720S,
)

# Check for sunpy availability
try:
    import astropy.units as u
    from sunpy.map import Map
    from sunpy.net import Fido, attrs as a
    HAS_SUNPY = True
except ImportError:
    HAS_SUNPY = False

print(f"sunpy installed: {HAS_SUNPY}")

## Helper Functions

In [ ]:
def create_synthetic_hmi_sequence(
    n_frames: int = 21,
    size: int = 256,
    cadence_seconds: float = 720.0,
    latitude_deg: float = 0.0,
) -> dict:
    """
    Create synthetic HMI-like magnetogram sequence with solar rotation.

    Parameters
    ----------
    n_frames : int
        Number of frames in sequence.
    size : int
        Image size (square).
    cadence_seconds : float
        Time between frames in seconds.
    latitude_deg : float
        Heliographic latitude of the feature.

    Returns
    -------
    dict
        Dictionary containing:
        - images: list of 2D numpy arrays
        - rsun_pixels: solar radius in pixels
        - cadence_hours: cadence in hours
        - latitude_deg: latitude
    """
    # Solar radius in pixels (typical HMI scale)
    rsun_pixels = size * 0.4

    # Calculate rotation rate
    rotation_rate = snodgrass_rotation_rate(latitude_deg)  # deg/day

    # Create coordinate grids
    y, x = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    center = size // 2

    # Solar disk mask
    r = np.sqrt((x - center)**2 + (y - center)**2)
    disk_mask = r < rsun_pixels

    # Create a bipolar active region pattern
    def create_bipole(x, y, cx, cy, separation=20, strength=500, width=15):
        """Create bipolar magnetic region."""
        # Positive polarity
        pos = strength * np.exp(-((x - cx - separation/2)**2 + (y - cy)**2) / (2 * width**2))
        # Negative polarity
        neg = -strength * np.exp(-((x - cx + separation/2)**2 + (y - cy)**2) / (2 * width**2))
        return pos + neg

    # Initial feature position
    feature_x0 = center - 30  # Start west of center
    feature_y = center + int(latitude_deg * rsun_pixels / 90)  # Approximate

    images = []
    cadence_hours = cadence_seconds / 3600.0

    for i in range(n_frames):
        # Calculate x shift due to rotation
        time_hours = i * cadence_hours
        x_shift = solar_rotation_shift(
            rsun_pixels=rsun_pixels,
            time_offset_hours=time_hours,
            latitude_deg=latitude_deg,
        )

        # Current feature position
        feature_x = feature_x0 + x_shift

        # Create magnetogram
        image = np.zeros((size, size), dtype=np.float32)

        # Add bipolar region
        bipole = create_bipole(x, y, feature_x, feature_y)
        image += bipole

        # Add quiet sun noise
        image += np.random.randn(size, size).astype(np.float32) * 5

        # Apply disk mask
        image = np.where(disk_mask, image, 0)

        images.append(image)

    return {
        "images": images,
        "rsun_pixels": rsun_pixels,
        "cadence_hours": cadence_hours,
        "latitude_deg": latitude_deg,
        "n_frames": n_frames,
    }

## 1. Snodgrass Differential Rotation Model

The Snodgrass (1983) model describes how solar rotation rate varies with latitude:

$$\omega(B) = A + B \cdot \sin^2(B) + C \cdot \sin^4(B)$$

where B is heliographic latitude.

In [ ]:
print(f"Constants:")
print(f"    A = {SNODGRASS_A:.2f} deg/day (equatorial rate)")
print(f"    B = {SNODGRASS_B:.2f} deg/day")
print(f"    C = {SNODGRASS_C:.2f} deg/day")
print(f"\nCarrington rotation period: {SOLAR_ROTATION_PERIOD:.2f} days (~26° latitude)")

In [ ]:
# Calculate rotation rates at different latitudes
print("Rotation rates at different latitudes:")
print("-" * 40)
print(f"{'Latitude':>10} | {'Rate (deg/day)':>15} | {'Period (days)':>13}")
print("-" * 40)

for lat in [0, 15, 30, 45, 60, 75]:
    rate = snodgrass_rotation_rate(lat)
    period = 360.0 / rate
    print(f"{lat:>10}° | {rate:>15.2f} | {period:>13.2f}")

In [ ]:
# Calculate pixel shift example
print("Pixel shift due to rotation (rsun=1600 pixels, 1 hour):")
print("-" * 40)
rsun = 1600  # Typical HMI solar radius

for lat in [0, 30, 60]:
    shift = solar_rotation_shift(rsun, time_offset_hours=1.0, latitude_deg=lat)
    print(f"   Latitude {lat:>2}°: {shift:>6.2f} pixels/hour")

## 2. StreamingStackAccumulator Demo

StreamingStackAccumulator uses Welford's online algorithm for numerically stable incremental computation of mean and variance. Memory usage is O(image_size), independent of number of images.

In [ ]:
# Create synthetic data
print("Creating synthetic sequence (21 frames, 128x128)...")
data = create_synthetic_hmi_sequence(n_frames=21, size=128)

# Initialize accumulator
shape = data["images"][0].shape
accumulator = StreamingStackAccumulator(shape)

print(f"\nAccumulating {len(data['images'])} images...")

for i, image in enumerate(data["images"]):
    accumulator.add(image)

    if (i + 1) % 7 == 0:
        print(f"   After {i+1} images: mean range = "
              f"[{accumulator.get_mean().min():.1f}, {accumulator.get_mean().max():.1f}]")

In [ ]:
# Get final statistics
mean = accumulator.get_mean()
std = accumulator.get_std()

print(f"Final statistics:")
print(f"   Images accumulated: {accumulator.count}")
print(f"   Mean range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"   Std range: [{std.min():.2f}, {std.max():.2f}]")

# Compare with numpy
numpy_mean = np.mean(data["images"], axis=0)
numpy_std = np.std(data["images"], axis=0, ddof=1)

print(f"\nComparison with numpy (should be nearly identical):")
print(f"   Max mean difference: {np.abs(mean - numpy_mean).max():.2e}")
print(f"   Max std difference: {np.abs(std - numpy_std).max():.2e}")

## 3. Rotation-Corrected Stacking

`stack_with_rotation_correction()` aligns images by compensating for solar rotation before combining them. This preserves spatial features that would otherwise blur due to rotation.

In [ ]:
# Create synthetic sequence at different latitudes
print("Creating sequences at different latitudes...")

for lat in [0, 30]:
    print(f"\n--- Latitude {lat}° ---")

    data = create_synthetic_hmi_sequence(
        n_frames=21,
        size=128,
        cadence_seconds=720.0,
        latitude_deg=lat,
    )

    images = data["images"]
    print(f"   Frames: {len(images)}, Size: {images[0].shape}")
    print(f"   Cadence: {data['cadence_hours']*60:.0f} minutes")

    # Stack without rotation correction (simple mean)
    simple_mean = np.mean(images, axis=0)

    # Stack with rotation correction
    stacked = stack_with_rotation_correction(
        images=images,
        rsun_pixels=data["rsun_pixels"],
        cadence_hours=data["cadence_hours"],
        crop_center=(64, 64),
        crop_size=64,
        latitude_deg=lat,
        method='mean',
    )

    print(f"   Simple mean shape: {simple_mean.shape}")
    print(f"   Rotation-corrected shape: {stacked.shape}")

    # Compare feature sharpness (proxy: max gradient)
    simple_grad = np.sqrt(np.gradient(simple_mean[32:96, 32:96])[0]**2 +
                          np.gradient(simple_mean[32:96, 32:96])[1]**2).max()
    corrected_grad = np.sqrt(np.gradient(stacked)[0]**2 +
                             np.gradient(stacked)[1]**2).max()

    print(f"   Feature sharpness (max gradient):")
    print(f"      Without correction: {simple_grad:.2f}")
    print(f"      With correction: {corrected_grad:.2f}")

## 4. Combining Methods Comparison

Available methods:
- `'list'` - Return all aligned images (no combining)
- `'mean'` - Simple arithmetic mean
- `'median'` - Robust median (outlier resistant)
- `'sigma_clipped'` - Iterative sigma clipping

In [ ]:
# Create data with outliers
print("Creating sequence with simulated cosmic rays...")
data = create_synthetic_hmi_sequence(n_frames=21, size=64)
images = data["images"]

# Add artificial outliers (cosmic ray hits)
np.random.seed(42)
for i in range(5):  # 5 frames with cosmic rays
    frame_idx = np.random.randint(0, len(images))
    y, x = np.random.randint(10, 54, size=2)
    images[frame_idx][y:y+3, x:x+3] = 2000  # Bright spike

# Stack with different methods
results = {}
for method in ['mean', 'median', 'sigma_clipped']:
    stacked = stack_with_rotation_correction(
        images=images,
        rsun_pixels=data["rsun_pixels"],
        cadence_hours=data["cadence_hours"],
        crop_center=(32, 32),
        crop_size=32,
        method=method,
        sigma_lower=3.0,
        sigma_upper=3.0,
    )
    results[method] = stacked

# Compare
print("\nResults (center 32x32 region):")
print("-" * 50)
print(f"{'Method':<15} | {'Max value':<12} | {'Std dev':<12}")
print("-" * 50)

for method, stacked in results.items():
    print(f"{method:<15} | {stacked.max():<12.2f} | {stacked.std():<12.2f}")

print("\nNote: Cosmic rays cause high max values in 'mean'.")
print("'median' and 'sigma_clipped' are more robust to outliers.")

## 5. Real HMI Data Download and Stacking

This section demonstrates how to download real HMI magnetogram data using sunpy Fido and apply rotation-corrected stacking.

**Requirements**: sunpy, astropy (`pip install sunpy astropy`)

In [ ]:
def download_hmi_data(
    start_time: datetime,
    n_frames: int = 11,
    cadence_minutes: int = 12,
    data_dir: str = "./data/hmi",
) -> list:
    """
    Download HMI LOS magnetogram data using sunpy Fido.

    Parameters
    ----------
    start_time : datetime
        Start time for data sequence.
    n_frames : int
        Number of frames to download.
    cadence_minutes : int
        Approximate cadence in minutes (12 for 720s magnetograms).
    data_dir : str
        Directory to save files.

    Returns
    -------
    list
        List of downloaded file paths.
    """
    if not HAS_SUNPY:
        raise ImportError("sunpy is required for data download. "
                         "Install with: pip install sunpy")

    data_path = Path(data_dir)
    data_path.mkdir(parents=True, exist_ok=True)

    # Calculate time range
    duration_minutes = n_frames * cadence_minutes
    end_time = start_time + timedelta(minutes=duration_minutes)

    print(f"Searching for HMI data from {start_time} to {end_time}...")

    # Search for HMI LOS magnetograms
    result = Fido.search(
        a.Time(start_time, end_time),
        a.Instrument("HMI"),
        a.Physobs("LOS_magnetic_field"),
    )

    if len(result) == 0 or len(result[0]) == 0:
        print("No data found!")
        return []

    # Limit to requested number of frames
    n_available = min(len(result[0]), n_frames)
    print(f"Found {len(result[0])} files, downloading {n_available}...")

    # Download
    files = Fido.fetch(result[0, :n_available], path=str(data_path), progress=True)

    return sorted([str(f) for f in files])

### 5.1 Download HMI Data

Download 11 frames of HMI LOS magnetograms (about 2 hours of data with 12-minute cadence).

In [ ]:
# Skip this cell if sunpy is not installed
if not HAS_SUNPY:
    print("sunpy not installed. Skipping real data download.")
    print("Install with: pip install sunpy astropy")
    hmi_files = []
else:
    # Download HMI data
    start_time = datetime(2024, 1, 15, 12, 0, 0)
    hmi_files = download_hmi_data(
        start_time=start_time,
        n_frames=11,
        cadence_minutes=12,
        data_dir="./data/hmi_stack",
    )
    print(f"\nDownloaded {len(hmi_files)} files")

### 5.2 Load and Inspect Data

Load the downloaded FITS files using sunpy Map and inspect their properties.

In [ ]:
if HAS_SUNPY and len(hmi_files) >= 3:
    # Load maps
    print("Loading HMI maps...")
    hmi_maps = [Map(f) for f in hmi_files]
    
    # Print info about first and last maps
    print(f"\nFirst map:")
    print(f"   Time: {hmi_maps[0].date}")
    print(f"   Shape: {hmi_maps[0].data.shape}")
    print(f"   RSUN_OBS: {hmi_maps[0].meta.get('RSUN_OBS', 'N/A')} arcsec")
    print(f"   CDELT1: {hmi_maps[0].meta.get('CDELT1', 'N/A')} arcsec/pixel")
    
    print(f"\nLast map:")
    print(f"   Time: {hmi_maps[-1].date}")
    
    # Calculate time span
    dt = (hmi_maps[-1].date - hmi_maps[0].date).to(u.hour)
    print(f"\nTotal time span: {dt:.2f}")
    print(f"Number of maps: {len(hmi_maps)}")
else:
    print("Skipping: No HMI files available or sunpy not installed.")

### 5.3 Prepare Data for Stacking

Extract image data, calculate solar radius in pixels, and compute cadence.

In [ ]:
if HAS_SUNPY and len(hmi_files) >= 3:
    # Extract images as numpy arrays
    hmi_images = [m.data.astype(np.float32) for m in hmi_maps]
    
    # Get solar radius in pixels from FITS header
    # RSUN_OBS is in arcsec, CDELT1 is arcsec/pixel
    rsun_obs = hmi_maps[0].meta.get('RSUN_OBS', 960.0)  # arcsec
    cdelt = abs(hmi_maps[0].meta.get('CDELT1', 0.5))    # arcsec/pixel
    rsun_pixels = rsun_obs / cdelt
    
    # Calculate cadence from timestamps
    t0 = hmi_maps[0].date.datetime
    t1 = hmi_maps[1].date.datetime
    cadence_seconds = (t1 - t0).total_seconds()
    cadence_hours = cadence_seconds / 3600.0
    
    print(f"Image shape: {hmi_images[0].shape}")
    print(f"Solar radius: {rsun_pixels:.1f} pixels")
    print(f"Cadence: {cadence_seconds:.0f} seconds ({cadence_hours*60:.1f} minutes)")
    print(f"Data range: [{hmi_images[0].min():.1f}, {hmi_images[0].max():.1f}] Gauss")
else:
    print("Skipping: No HMI files available or sunpy not installed.")

### 5.4 Apply Rotation-Corrected Stacking

Stack the images with solar rotation correction. We'll crop a 512x512 region from the center and compare results with and without rotation correction.

In [ ]:
if HAS_SUNPY and len(hmi_files) >= 3:
    # Stacking parameters
    crop_size = 512
    center_y, center_x = hmi_images[0].shape[0] // 2, hmi_images[0].shape[1] // 2
    
    print(f"Stacking {len(hmi_images)} images...")
    print(f"Crop center: ({center_y}, {center_x})")
    print(f"Crop size: {crop_size}x{crop_size}")
    
    # Stack WITHOUT rotation correction (simple mean)
    print("\n1. Simple mean (no rotation correction)...")
    half = crop_size // 2
    y0, y1 = center_y - half, center_y + half
    x0, x1 = center_x - half, center_x + half
    cropped_images = [img[y0:y1, x0:x1] for img in hmi_images]
    simple_mean = np.mean(cropped_images, axis=0)
    print(f"   Shape: {simple_mean.shape}")
    print(f"   Range: [{simple_mean.min():.1f}, {simple_mean.max():.1f}] Gauss")
    
    # Stack WITH rotation correction
    print("\n2. Rotation-corrected stacking...")
    stacked_mean = stack_with_rotation_correction(
        images=hmi_images,
        rsun_pixels=rsun_pixels,
        cadence_hours=cadence_hours,
        crop_center=(center_y, center_x),
        crop_size=crop_size,
        latitude_deg=0.0,  # Equatorial region
        method='mean',
    )
    print(f"   Shape: {stacked_mean.shape}")
    print(f"   Range: [{stacked_mean.min():.1f}, {stacked_mean.max():.1f}] Gauss")
    
    # Stack with median (robust to outliers)
    print("\n3. Rotation-corrected stacking (median)...")
    stacked_median = stack_with_rotation_correction(
        images=hmi_images,
        rsun_pixels=rsun_pixels,
        cadence_hours=cadence_hours,
        crop_center=(center_y, center_x),
        crop_size=crop_size,
        latitude_deg=0.0,
        method='median',
    )
    print(f"   Shape: {stacked_median.shape}")
    print(f"   Range: [{stacked_median.min():.1f}, {stacked_median.max():.1f}] Gauss")
else:
    print("Skipping: No HMI files available or sunpy not installed.")

### 5.5 Compare Results

Compare the feature sharpness between stacking methods using gradient magnitude as a proxy.

In [ ]:
if HAS_SUNPY and len(hmi_files) >= 3:
    def compute_sharpness(image):
        """Compute image sharpness using gradient magnitude."""
        gy, gx = np.gradient(image)
        grad_mag = np.sqrt(gx**2 + gy**2)
        return grad_mag.mean(), grad_mag.max()
    
    # Compare sharpness
    print("Sharpness Comparison (gradient magnitude):")
    print("-" * 60)
    print(f"{'Method':<30} | {'Mean grad':<12} | {'Max grad':<12}")
    print("-" * 60)
    
    methods = [
        ("Simple mean (no correction)", simple_mean),
        ("Rotation-corrected (mean)", stacked_mean),
        ("Rotation-corrected (median)", stacked_median),
    ]
    
    for name, img in methods:
        mean_grad, max_grad = compute_sharpness(img)
        print(f"{name:<30} | {mean_grad:<12.2f} | {max_grad:<12.2f}")
    
    # Statistics comparison
    print("\n\nStatistics Comparison:")
    print("-" * 60)
    print(f"{'Method':<30} | {'Std dev':<12} | {'MAD':<12}")
    print("-" * 60)
    
    for name, img in methods:
        std = img.std()
        mad = np.median(np.abs(img - np.median(img)))
        print(f"{name:<30} | {std:<12.2f} | {mad:<12.2f}")
    
    print("\nNote: Higher gradient values indicate sharper features.")
    print("Rotation correction should preserve feature sharpness better than simple stacking.")
else:
    print("Skipping: No HMI files available or sunpy not installed.")

## Typical Usage Patterns

### High-level API: Stacking class (requires sunpy)

```python
from egghouse.sdo import Stacking

# Create stacker
stacker = Stacking(
    nb_stack=21,           # Number of images to stack
    crop_size=512,         # Output size
    method='mean',         # 'mean', 'median', or 'sigma_clipped'
    latitude_deg=0.0,      # For differential rotation
)

# Run on FITS files
file_paths = ['hmi_001.fits', 'hmi_002.fits', ...]
result = stacker.run(file_paths)
```

### Low-level API: stack_with_rotation_correction (numpy arrays)

```python
from egghouse.sdo import stack_with_rotation_correction

# Prepare images as numpy arrays
images = [...]  # List of 2D arrays

# Stack with rotation correction
stacked = stack_with_rotation_correction(
    images=images,
    rsun_pixels=1600,      # Solar radius in pixels
    cadence_hours=0.2,     # 12 minutes
    crop_center=(2048, 2048),
    crop_size=512,
    latitude_deg=15.0,
    method='sigma_clipped',
)
```

### Memory-efficient streaming accumulation

```python
from egghouse.sdo import StreamingStackAccumulator

accumulator = StreamingStackAccumulator(shape=(4096, 4096))

for fits_file in large_file_list:
    image = load_fits(fits_file)
    # Apply rotation correction here if needed
    accumulator.add(image)

mean = accumulator.get_mean()
std = accumulator.get_std()
```

### Snodgrass rotation calculation

```python
from egghouse.sdo import snodgrass_rotation_rate, solar_rotation_shift

# Get rotation rate at latitude
rate = snodgrass_rotation_rate(latitude_deg=30)  # deg/day

# Calculate pixel shift
shift = solar_rotation_shift(
    rsun_pixels=1600,
    time_offset_hours=2.0,  # 2 hours from reference
    latitude_deg=30,
)
```